# Churn Analysis

In the previous exploratory analysis, several variables exhibited meaningful
associations with customer churn, including tenure, pricing, internet
services, payment behavior, and service adoption.

This notebook expands the analysis by investigating interactions between
variables, creating derived features, and identifying broader customer
behavior patterns associated with churn.

In [ ]:
import pandas as pd
from src import (
    load_customers,
    plot_line,
    plot_histogram,
    plot_heatmap)

df = load_customers()

df.head()

---
## Feature Engineering

In this section, we create derived variables intended to consolidate patterns identified during the exploratory analysis and facilitate interaction-based investigations.

Some engineered features, such as service count, are directly analyzed due to their analytical relevance and ability to summarize multiple service adoption patterns. Other features are created primarily to support later interaction analyses and customer segmentation.

### service_count

To consolidate the relationship previously observed between additional service adoption and churn behavior, we create a feature representing the number of additional internet-related services subscribed by each customer.

In [ ]:
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["service_count"] = (
    df[service_columns] == "Yes"
).sum(axis=1)

df["service_count"].value_counts().sort_index()

In [ ]:
plot_histogram(
    df=df,
    column="service_count",
    bins=7,
    title="Customer Distribution by Service Count",
    xlabel="Number of Additional Services",
    ylabel="Customers"
)

The distribution reveals a high concentration of customers with no additional internet-related services, followed by a sharp decline for customers with only one service.

After this initial decline, customers become more distributed across intermediate service counts, with a secondary concentration around three additional services.

In [ ]:
service_count_churn = (
    df.groupby("service_count")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="churn_rate")
)

service_count_churn

In [ ]:
plot_line(
    df=service_count_churn,
    x="service_count",
    y="churn_rate",
    title="Churn Rate by Service Count",
    xlabel="Number of Additional Services",
    ylabel="Churn Rate (%)"
)

Customers with no internet-related services present a churn rate of 21.41%. After the acquisition of a single additional service, churn rises sharply to 45.76%, then decreases consistently as more services are added.

Customers with six additional services exhibit the lowest churn rate, at only
5.28%.

This pattern reinforces the distinction between different customer profiles previously observed during the exploratory analysis. Customers with broader service adoption tend to exhibit substantially stronger retention behavior.

It is important to note that customers with no internet-related services are not necessarily the same as customers without internet service. Customers may still subscribe to internet plans while not adopting additional services such as online security or streaming features.

This distinction explains the difference between the 7.40% churn rate observed among customers without internet service and the 21.41% churn rate observed among customers without additional internet-related services.

### Tenure Groups

To facilitate interaction-based analyses and reduce noise from individual
tenure values, we group customers into broader tenure categories representing
different stages of the customer lifecycle.

In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"],
    include_lowest=True
)

df["tenure_group"].value_counts().sort_index()

### Monthly Charge Groups

To simplify interaction-based analyses and reduce volatility from individual monthly charge values, customers are grouped into broader pricing categories.

In [ ]:
df["monthly_charge_group"] = pd.cut(
    df["MonthlyCharges"],
    bins=[15, 25, 35, 50, 65, 80, 95, 110, 120],
    labels=[
        "15-24",
        "25-34",
        "35-49",
        "50-64",
        "65-79",
        "80-94",
        "95-109",
        "110+"
    ],
    include_lowest=True
)

df["monthly_charge_group"].value_counts().sort_index()

---
## Interaction Analysis

### Service Count and Tenure

Previous analyses suggested that both customer tenure and service adoption are strongly associated with churn behavior.

We now investigate how these variables interact in order to better understand whether the relationship between service adoption and churn may be partially influenced by customer lifetime.

In [ ]:
service_tenure_count = pd.crosstab(
    df["service_count"],
    df["tenure_group"]
)

service_tenure_count

In [ ]:
service_tenure_churn = pd.crosstab(
    df["service_count"],
    df["tenure_group"],
    values=(df["Churn"] == "Yes"),
    aggfunc="mean"
) * 100

service_tenure_churn

In [ ]:
plot_heatmap(
    df=service_tenure_churn,
    title="Churn Rate by Service Count and Tenure Group",
    xlabel="Tenure Group",
    ylabel="Service Count"
)

Across all service count categories, churn rates decrease substantially as customer tenure increases, reinforcing the strong relationship previously observed between tenure and customer retention.

The effect of service adoption remains present within individual tenure groups, although the relationship becomes less consistent compared to the isolated analysis performed earlier.

This suggests that part of the relationship previously observed between service adoption and lower churn may be associated with customer lifetime. Customers with longer tenure may naturally accumulate more subscribed services over time while simultaneously exhibiting lower churn behavior.

Some extreme values should be interpreted cautiously due to very small sample sizes in combinations involving high service counts and low tenure.

### Monthly Charges and Tenure

The exploratory analysis previously suggested that churn risk may be particularly elevated among customers combining high monthly charges and low tenure.

We now formalize this relationship through grouped interaction analysis in order to quantify how churn behavior varies across customer lifecycle and pricing segments.

In [ ]:
charge_tenure_count = pd.crosstab(
    df["monthly_charge_group"],
    df["tenure_group"]
)

charge_tenure_count

In [ ]:
charge_tenure_churn = pd.crosstab(
    df["monthly_charge_group"],
    df["tenure_group"],
    values=(df["Churn"] == "Yes"),
    aggfunc="mean"
) * 100

charge_tenure_churn

In [ ]:
plot_heatmap(
    df=charge_tenure_churn,
    title="Churn Rate by Monthly Charge Group and Tenure Group",
    xlabel="Tenure Group",
    ylabel="Monthly Charge Group"
)

Across nearly all monthly charge groups, churn rates decrease substantially as customer tenure increases, reinforcing the strong relationship previously observed between tenure and customer retention.

The effect of monthly charges appears considerably stronger among low-tenure customers. High-charge customers in the early stages of the customer lifecycle present extremely elevated churn rates, whereas the same pricing ranges become substantially more stable among long-tenure customers.

This pattern suggests that customer sensitivity to pricing may decrease as customers remain longer with the company, indicating an important interaction between pricing and customer maturity.

Some extreme values should be interpreted cautiously due to very small sample sizes in combinations involving very high charges and low tenure.

Nevertheless, the 95-109 charge range consistently presents elevated churn rates even among higher-tenure customers, where sample sizes are substantially more stable. This suggests that customers within this pricing segment may represent a particularly high-risk group relative to neighboring pricing ranges.